# Completo

In [1]:
import numpy as np
import scipy
from scipy.signal import find_peaks, welch
from lib.readwav import *
import matplotlib.pyplot as plt

In [2]:
fundamentals = [] # Array contenente le sole fondamentali per ciascun file
harmonics = [] # Lista contenente le armoniche: ogni riga è un file, ogni colonna un'armonica (inclusa la fondamentale)
power_spectra = [] # Lista contenente gli spettri valutati nelle frequenze delle armoniche
normalized_power_spectra = [] # Lista contenente sempre gli spettri valutati, ma ora anche divisi per la potenza della fondamentale

In [3]:
# troviamo il massimo comun divisore delle differenze
def f0_MCD(picchi, limite):
    minimo = np.min(picchi) / 2
    massimo = limite
    # Risoluzione 0.1 Hz
    candidati = np.arange(minimo, massimo + 0.1, 0.1)
    
    # dall'alto verso il basso
    candidati = candidati[::-1]

    residuo_minimo = float('inf')
    migliore_f0 = minimo

    for f0 in candidati:
        rapporto = picchi / f0
        interi = np.round(rapporto)
        
        if np.any(interi == 0):
            continue

        # distanza tra i rapporti e gli interi
        distanze = np.abs(rapporto - interi)
        residuo_base = np.sum(distanze)

        armoniche_coperte = len(np.unique(interi))
        massima_armonica = np.max(interi)
        
        vuoti = massima_armonica - armoniche_coperte
        penalita_vuoti = vuoti * 0.1  # Peso regolabile

        residuo_attuale = residuo_base + penalita_vuoti

        # uso un margine per non cambiare f0 se il miglioramento è minimo
        if residuo_attuale < residuo_minimo * 0.95:
            residuo_minimo = residuo_attuale
            migliore_f0 = f0

    return migliore_f0

In [6]:
for file in range(280):
    #print("È in corso l'analisi del file",file+1,"...")
    # Lettura file:
    filename = "./wav_files/note_{}.wav".format(file+1)
    rate, note = readwav(filename)
    note = note[:,0]
    start = int(len(note) * 0.25)
    end = int(len(note) * 0.9)
    note = note[start:end]
    time = np.arange(len(note)) / rate

    # applicazione welch
    sigma = 10000
    gaussian_window = scipy.signal.windows.gaussian(len(note)//10, std=sigma)
    frequencies, spectrum = welch(note, fs=rate, nperseg=len(gaussian_window), window=gaussian_window)
    log_frequencies = np.log10(frequencies[1:])
    log_spectrum = np.log10(spectrum[1:])

    # calcolo picchi
    prom = 2.2
    peaks, _ = find_peaks(log_spectrum, prominence=prom)

    # calcolo f0
    #prendo tra i picchi quello che ha il power spectrum massimo
    index_max_peak = np.argmax(log_spectrum[peaks])
    freq_limite = 10**log_frequencies[peaks][index_max_peak]
    frequencies_peaks = 10**log_frequencies[peaks]

    f0 = f0_MCD(frequencies_peaks[:10], freq_limite)
    fundamentals.append(f0)

    # stima armoniche
    armoniche_stimate = { f0*i  for i in range(1,20)}
    

    # power spectrum alle armoniche stimate

    power_armoniche = []
    armoniche_stimate = sorted(list(armoniche_stimate))
    harmonics.append(armoniche_stimate)

    tolleranza_picco = 0.04 
    for armonica in armoniche_stimate:
        log_armonica = np.log10(armonica)
        # Trova se c'è un picco trovato vicino all'armonica
        distanza_picchi = np.abs(log_frequencies[peaks] - log_armonica)
        if np.any(distanza_picchi < tolleranza_picco):
            # Se c'è un picco vicino, prendo il valore del picco
            indice_picco = peaks[np.argmin(distanza_picchi)]
            power_armoniche.append(log_spectrum[indice_picco])
        else:
            # Altrimenti prendo il valore più vicino
            distanza = np.abs(log_frequencies - log_armonica)
            indice_vicino = np.argmin(distanza)
            power_armoniche.append(log_spectrum[indice_vicino])
    
    power_f0 = power_armoniche[0]
    
    # normalizzo i valori di potenza delle armoniche rispetto alla potenza del picco fondamentale
    power_armoniche = np.array(power_armoniche)
    power_armoniche_normalizzate = power_armoniche - power_f0

    power_spectra.append(power_armoniche)
    normalized_power_spectra.append(power_armoniche_normalizzate)
    
    # visualizzo valori come dei puntini blu sul grafico del power spectrum
    #plt.figure(figsize=(12, 6))
    #plt.plot(log_frequencies, log_spectrum)
    #plt.scatter(np.log10(armoniche_stimate), power_armoniche, color='blue')
    #for armonica in armoniche_stimate:
    #    plt.axvline(np.log10(armonica), color='red', linestyle='--')
    #plt.xlabel('Frequency in log scale (Hz)')
    #plt.ylabel('Power in log scale')
    #plt.grid()
    #plt.show()


In [7]:
# Salvataggio su file esterni:
np.savetxt("fundamentals.csv", fundamentals, delimiter = ",")
np.savetxt("harmonics.csv", harmonics, delimiter = ",")
np.savetxt("power_spectra.csv", power_spectra, delimiter = ",")
np.savetxt("normalized_power_spectra.csv", normalized_power_spectra, delimiter = ",")